# Round-1 Annotation Pipeline — Kaggle Runner

**Model**: `Qwen/Qwen3.5-9B` 

## Before running — checklist

| # | Step | Where |
|---|------|-------|
| 1 | Enable **GPU T4** | Session options → Accelerator → GPU T4 |
| 2 | Enable **Internet** | Session options → Internet → On |
| 3 | Add `HF_TOKEN` secret | Add-ons → Secrets → New secret |
| 4 | Attach your dataset | Add data → Your datasets |
| 5 | Set `REPO_URL` and `DATASET_PATH` in **Cell 4** | — |

> **Internet must be ON** — the notebook clones the repo from GitHub and downloads  

The pipeline saves a checkpoint after every batch — re-running the notebook resumes from where it stopped.

## 1. Install dependencies

`Qwen3.5-9B` requires `transformers >= 4.51.0`. We install from the GitHub source to get the latest stable version.

In [ ]:
import subprocess, sys

def _pip(*args):
    cmd = [sys.executable, "-m", "pip", "install", "-q"] + list(args)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-2000:] if r.stdout else "")
        raise RuntimeError(f"pip install failed:\n{r.stderr[-2000:]}")

_pip("git+https://github.com/huggingface/transformers")  # >= 4.51.0 for Qwen3.5-2B
_pip("accelerate>=0.30.0", "bitsandbytes>=0.43.0",
     "pydantic>=2.0", "pyyaml", "tqdm", "Pillow")
print("All dependencies installed.")

## 2. Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU detected. Enable GPU in Session options.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU  : {gpu_name}")
print(f"VRAM : {vram_gb:.1f} GB")

import shutil
free_gb = shutil.disk_usage("/kaggle/working").free / 1024**3
print(f"Disk free (/kaggle/working) : {free_gb:.1f} GB")
if free_gb < 6:
    print("WARNING: Less than 6 GB free — model download may fail.")

## 3. Verify transformers version

In [ ]:
import transformers
print(f"Transformers version : {transformers.__version__}")

try:
    from transformers import AutoModelForImageTextToText
    print("AutoModelForImageTextToText : OK")
except ImportError as e:
    raise RuntimeError(
        f"AutoModelForImageTextToText not found: {e}\n"
        "Re-run Cell 1 to reinstall transformers from source."
    )

## 4. Configuration

**Edit these variables before running.**

| Variable | Description |
|----------|-------------|
| `REPO_URL` | GitHub repo URL to clone |
| `DATASET_SLUG` | Kaggle dataset slug (owner/dataset-name) |
| `MAX_RECORDS` | **Test mode**: set to e.g. `5` to run only the first 5 records; set `None` to run all |

In [ ]:
import os
from pathlib import Path

# ---------------------------------------------------------------------------
# EDIT HERE
# ---------------------------------------------------------------------------
REPO_URL     = "https://github.com/nhatvu205/vi-multimodal-sacarsm-detection-on-social-media.git"
DATASET_NAME = "datasets/nhatvu205/sacasm-dataset-uit/merged_dataset"  # owner/dataset-name

# Test mode: True  → dùng 50_samples.json + 50_samples_images/ (50 mẫu cố định)
#            False → dùng data.json + images/ (toàn bộ dataset)
TEST_MODE = True
# ---------------------------------------------------------------------------

KAGGLE_WORKING = Path("/kaggle/working")
REPO_DIR       = KAGGLE_WORKING / "vi-multimodal-sacarsm-detection-on-social-media"
ROUND1_DIR     = REPO_DIR / "round-1-annotation"
OUTPUT_DIR     = KAGGLE_WORKING / ("outputs_test" if TEST_MODE else "outputs")

# Kaggle mounts datasets at /kaggle/input/<dataset-name>/ (name only, no owner prefix)
PKG_DIR = Path("/kaggle/input") / DATASET_NAME

if TEST_MODE:
    INPUT_JSON = PKG_DIR / "50_samples.json"
    IMAGES_DIR = PKG_DIR / "50_samples_images"
else:
    INPUT_JSON = PKG_DIR / "data.json"
    IMAGES_DIR = PKG_DIR / "images"

HF_CACHE_DIR = KAGGLE_WORKING / "hf_cache"
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_CACHE_DIR)

mode_label = f"TEST MODE (50_samples.json + 50_samples_images/)" if TEST_MODE else "FULL MODE (data.json)"
print(f"Repo URL     : {REPO_URL}")
print(f"Dataset path : {PKG_DIR}")
print(f"Input JSON   : {INPUT_JSON}")
print(f"Images dir   : {IMAGES_DIR}")
print(f"Output dir   : {OUTPUT_DIR}")
print(f"Running      : {mode_label}")

## 5. Sanity-check dataset

In [ ]:
import json

if not INPUT_JSON.exists():
    raise FileNotFoundError(f"Input JSON not found: {INPUT_JSON}")
if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"Images folder not found: {IMAGES_DIR}")

raw = INPUT_JSON.read_text(encoding="utf-8").strip()
all_records = json.loads(raw) if raw.startswith("[") else [json.loads(l) for l in raw.splitlines() if l.strip()]

print(f"Input JSON   : {INPUT_JSON}")
print(f"Records      : {len(all_records)}")
print(f"Images dir   : {IMAGES_DIR}")
print(f"Images found : {len(list(IMAGES_DIR.iterdir()))}")

## 6. Clone repo and set Python path

In [ ]:
import subprocess, sys

REPO_BRANCH = "feat/round-1-annotation"

if REPO_URL and not REPO_DIR.exists():
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH}) ...")
    r = subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{r.stderr}")
    print("Clone complete.")
elif REPO_DIR.exists():
    print(f"Repo present at {REPO_DIR} — pulling latest ...")
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)

if str(ROUND1_DIR) not in sys.path:
    sys.path.insert(0, str(ROUND1_DIR))
print(f"sys.path includes: {ROUND1_DIR}")

## 7. Load HuggingFace token

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("HF_TOKEN loaded from Kaggle Secrets.")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")
    if HF_TOKEN:
        print("HF_TOKEN loaded from environment variable.")
    else:
        print("HF_TOKEN not set — will proceed without it (public models only).")

## 8. Patch image paths and write input file

In [ ]:
import src.llm_judge as lj
from PIL import Image
from typing import Optional

# Rewrite image_path in every record to the absolute Kaggle path
for record in all_records:
    if "image_path" in record:
        record["image_path"] = str(IMAGES_DIR / Path(record["image_path"]).name)

# Patch _open_image to resolve images from IMAGES_DIR directly
def _open_image_patched(image_path: str, max_pixels: int = 500_000) -> Optional[Image.Image]:
    candidate = IMAGES_DIR / Path(image_path).name
    if candidate.exists():
        return lj._resize_image(Image.open(candidate).convert("RGB"), max_pixels)
    lj.logger.warning("Image not found: %s", candidate)
    return None

lj._open_image = _open_image_patched

# Quick sanity-check on first record
img = lj._open_image(all_records[0]["image_path"])
print(f"Image patch OK — first record image size: {img.size if img else 'FAILED'}")

# Write patched JSON so the pipeline can read it
PATCHED_JSON = KAGGLE_WORKING / "patched_input.json"
PATCHED_JSON.write_text(json.dumps(all_records, ensure_ascii=False), encoding="utf-8")
print(f"Patched input written to: {PATCHED_JSON}")

## 9. Verify prompt files

In [ ]:
for fname in ["prompt.txt", "few-short-examples.txt"]:
    p = ROUND1_DIR / "prompts" / fname
    status = "OK" if p.exists() else "MISSING"
    print(f"{fname:30s} : {status}")
    if not p.exists():
        raise FileNotFoundError(f"Prompt file missing: {p}")

## 10. Smoke-test — model loading (optional)

Load `Qwen3.5-9B`, run one text inference, then free GPU memory.  
**Skip this cell** if you want to go straight to the pipeline.

In [ ]:
# from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
# import torch

# MODEL_ID = "Qwen/Qwen3.5-9B"
# print(f"Loading {MODEL_ID} in 4-bit ...")

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_compute_dtype=torch.float16,
# )
# smoke_model = AutoModelForImageTextToText.from_pretrained(
#     MODEL_ID, quantization_config=bnb_config, device_map="auto",
#     token=HF_TOKEN or None,
# )
# smoke_proc = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN or None)

# test_msgs = [{"role": "user", "content": [{"type": "text", "text": "Reply with the single word: OK"}]}]
# try:
#     inputs = smoke_proc.apply_chat_template(
#         test_msgs, tokenize=True, add_generation_prompt=True,
#         return_dict=True, return_tensors="pt", enable_thinking=False,
#     )
# except TypeError:
#     inputs = smoke_proc.apply_chat_template(
#         test_msgs, tokenize=True, add_generation_prompt=True,
#         return_dict=True, return_tensors="pt",
#     )

# device = next(smoke_model.parameters()).device
# inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}

# with torch.no_grad():
#     out_ids = smoke_model.generate(**inputs, max_new_tokens=10, do_sample=False)

# reply = smoke_proc.decode(out_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
# print(f"Smoke-test reply: '{reply}'")

# del smoke_model, smoke_proc, inputs, out_ids
# torch.cuda.empty_cache()
# print("Model unloaded — GPU memory released for pipeline.")

## 11. Run the annotation pipeline

Uses `MAX_RECORDS` from **Cell 4**:  
- `MAX_RECORDS = 5` → chạy 5 records đầu (test mode)  
- `MAX_RECORDS = None` → chạy toàn bộ dataset

In [ ]:
from src.pipeline_round1 import run_pipeline

CONFIG_PATH = ROUND1_DIR / "configs" / "round1.yaml"

mode_label = "TEST MODE (50_samples)" if TEST_MODE else "FULL MODE"
print("=" * 60)
print(f"Round-1 annotation pipeline ({mode_label})")
print(f"  Model  : Qwen/Qwen3.5-9B (4-bit)")
print(f"  Input  : {PATCHED_JSON} ({len(all_records)} records total)")
print(f"  Config : {CONFIG_PATH}")
print(f"  Output : {OUTPUT_DIR}")
print("=" * 60)

run_pipeline(
    input_data=str(PATCHED_JSON),
    config_path=str(CONFIG_PATH),
    output_dir=str(OUTPUT_DIR),
    hf_token=HF_TOKEN or None,
)

## 12. Summary

In [ ]:
import json
from pathlib import Path

stats_path = Path(OUTPUT_DIR) / "round1_stats.json"
if stats_path.exists():
    stats = json.loads(stats_path.read_text(encoding="utf-8"))
    print(json.dumps(stats, indent=2, ensure_ascii=False))
else:
    print(f"Stats file not found: {stats_path}")